## Perceptron

-- The majority of the course materials are adopted from CMSE 202 course materials, Michigan State University.

### 1. The Perceptron Model

The perceptron is one of the first used examples of what has come to be called a neural network. Invented in 1958, it was originally hailed as a way to achieve what had come to be called "Artificial Intelligence". However, it was quickly proved that the perceptron model was limited. The claims and subsequent refutations halted neural network research for a number of years.

Perceptrons are used as a kind of **classifier** that we can train using examples. The simplest perceptron is what is known as a binary classifier. By this we mean that, we can provide individual examples of two classes (hence, a "binary" classifier) where the individuals are represented by some number of features/inputs. All examples use the same input features, but the particular feature values are used for the classification process. The goal is the create the classifier such that, when a never-seen-before individual is provided to the perceptron, it can correctly determine that individual's class

There are ways to extend the perceptron's ability to deal with multiple classes (to classify the inputs as representing one of `n` classes instead of only two), but for this exercise we will only be concerned with a binary classifer.

The limitations of a binary perceptron are that the classes must be **linearly separable**. It is easier to show than to explain. Look at the two graphs below. The axes represent the range of values for the two input features. The dots represent individual input examples based on their corresponding feature values and the colors represent the class that each individual example belongs to. 

<img src="https://i.imgur.com/pU70IHB.png">

For experiment A, it is clear that we can draw a line through the 2D input/feature space such that we can separate the examples of the two classes. For experiment B, no such line separating the two classes exists. Furthermore, it is also clear that we could draw **many** lines for A such that we separate the two classes. 

**The limitations therefore are**:
- a perceptron can only classify elements that are linearly separable
- a perceptron cannot distinguish which linear boundary is "better".

### 2. It's a line

The way to think about a perceptron is that it is learning a hyperplane (in 2D, which is a line) through the feature space. A line is just a simple equation with the following form:

$$w_0 + w_1 x_1 + w_2 x_2 = 0,$$

where $w_i$ are weights and $x_i$ are features.. 

### Loading and inspecting the data

Before we build a machine learning model, we need data to base it off of. The data set we are going to use is available from the course GitHub repository and is called, binary-iris.csv.

You can download it from: https://raw.githubusercontent.com/msu-cmse-courses/cmse202-supplemental-data/main/data/binary-iris.csv

This file is a variation on a classic, simple classification data set for iris flowers from 1936. It is used often as a very simple test of learning systems. The original (see https://en.wikipedia.org/wiki/Iris_flower_data_set ) has 4 classes and 4 inputs, for our experiments we have modified the file to have only 2 inputs and 2 classes.

✅ Do This: Load the data into python and visualize (with a plot) to get a sense for what it looks like. Use different colors to represent the two different iris classifications.

In [ ]:
!Curl -O https://raw.githubusercontent.com/msu-cmse-courses/cmse202-supplemental-data/main/data/binary-iris.csv

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd

iris_df = pd.read_csv("binary-iris.csv")

iris_targets = iris_df.values[:,-1].astype(str) 
iris_data = iris_df.values[:,:-1].astype(float)

plt.title("Iris Dataset")
plt.xlabel("Sepal Length")
plt.ylabel("Sepal Width")

for species in ["Iris-setosa","Iris-versicolor"]:
    these_points = iris_data[ iris_targets == species ]
    plt.scatter(these_points[:,0], these_points[:,1], label = species)
    
plt.legend()

plt.show()

### 3. Build a Perceptron class 
Let's build a perceptron class. We created a "skeleton" class for you to fill in.

&#9989; **Do This:** Complete these two classes based on the specifications below:

- an `__init__` method. Basically every class needs this to set instance attributes (i.e. the variables attached to `self` -- make sure you're using the `self.<variable>` notation!). **The `__init__` method should:**
  - Take, as an argument, the labeled data (the two data features plus a data label, -1 or 1, at the end)
  - Create an object attribute `data` for the data passed in from the file. It should contain 2D vector where each row represents a sample (e.g. an iris flower). Each row should consists of two features and a label. The labels should be modified for 1 (iris setosa) and -1 (iris versicolor). 
  - Create a 1D weight vector attribute, `weights`, of the same shape as the number of features + 1. The **"+ 1"** extra term is for the **bias weight**. For now, fill the weights with some initial value (1.0 would be resonable choice to start). We'll fix that later. Remember that in our Perceptron model from above, the **bias weight term is the first value in the list of weights**.

- The class needs a `predict` method which takes a single argument, an array of features from a **single sample** in the data set. **It should do the following:**
   - multipy (*as a dot product*) the argument feature vector and the weights for those features.
   - add the bias weight (the first value in our list of weights) to the result.
   - apply the activation function to the result to say whether the label of that input is -1 vs 1
   - return that predicted class value

In [ ]:
import numpy as np

class Perceptron():

    def __init__ (self, labeled_data):        
        self.data = np.array(labeled_data)
        self.weights = np.full( (self.data.shape[1]), 1.0) # note that this is just an easy way to get "number of features + 1" and set them to 1.0
 
    def predict(self, feature_set):
        res = np.dot(feature_set, self.weights[1:])
        res += self.weights[0]
        return (1 if res > 0 else -1)

&#9989; **Do This:** Run the following code to test your new Perceptron class. See if it works, even though it is not yet a very good classifier. Iterate on your class until the output from this cell makes sense.

In [ ]:
## get data from file, just using file and string ops
f = open("binary-iris.csv")
header = next(f) # dump the header line
data = []
for line in f:
    fields = line.split(",")
    # need to strip label because, as the last element, it has a \n
    label = (1.0 if fields[2].strip() == "Iris-setosa" else -1.0)
    # the fields are strings until we conver them
    data.append([float(fields[0]), float(fields[1]), label])
f.close()

p = Perceptron(data)
print("weights: ", p.weights[1:])
print("bias weight:", p.weights[0])
print("prediction: ",p.predict([1,1])) # some arbitrary feature vector, just testing here
print(p.data)

### Learning

OK, now the interesting part. **We need to learn the value of the weights, including the value of the bias weight, so that the predictions of the `predict` method are good**. How do we do that?

The basic idea is this. We feed in the data that we have where we know the labels (and we do) to `predict`. We then compare the classification we want (from the existing data) and the classification we got (from the `predict` method). We use that difference to update **all the weights**. We do this for **each** of the data. 

However, we need to do one more thing. We need to not **over correct** the weights. If we do that then the weight values might swing wildly over time and never settle down. So we also provide a `learning_rate`. This rate reduces how much the weight changes. Overall then we use the following equation:

$$ self.weights[i] = self.weights[i] +  (self.learning\_rate * (class\_label - prediction\_label) * feature[i]) $$

We do this for each feature/input of the example and each corresponding weight. We also update the bias weight each time we update a weight. We use the same difference, `class_label-prediction_label`, for that update and, remember, the feature value for the bias is just "1" (from above). So, if `weights[0]` is the bias weight, then the update equation for the bias weight would be

$$ self.weights[0] = self.weights[0] + learning\_rate * (class\_label - prediction\_ label) $$

We do this for some number of iterations because, with a small `learning_rate`, we need to repeat the process to get the weight values correctly set.

### Modify our Perceptron class

We need to update our `Perceptron` class to deal with learning.

&#9989; **Do This:** Copy the class you wrote from above in the cell below and **make the following changes**:

- `__init__`, besides labeled data, should now take two more arguments: the **number of iterations of learning** you want and the **learning_rate**. These should be also stored as object attributes.
- Add a new method called `fit`. It should perform the number of iterations of learning provided in `__init__` on all the labeled data as described.
- Add a new method called `errors`. It should print out the number of errors for the predicted vs actual class labels and the current weights.

In [ ]:
import numpy as np

class Perceptron():

    def __init__ (self, labeled_data, iters, learning_rate):        
        self.data = np.array(labeled_data)
        self.weights = np.full((self.data.shape[1]), 1.0) # note that this is just an easy way to get "number of features + 1" and set them to 1.0
        self.iters = iters
        self.l_r = learning_rate
        # self.weight_history = []   # store weights over time       
        
    def predict(self, feature_set):
        # print("Features: ",feature_set, " weights: ", self.weights[1:])
        res = np.dot(feature_set, self.weights[1:])
        res += self.weights[0]
        return (1 if res > 0 else -1)
    
    def fit(self):
        for _ in range(self.iters):
            for row in self.data:
                update = self.l_r * (row[-1] - self.predict(row[:-1]))
                self.weights[1:] += update * row[:-1]
                self.weights[0] += update
            # save weights after each epoch
            # self.weight_history.append(self.weights.copy())

    def errors(self):
        error_count = 0
        for row in self.data:
            # print("data: ", row[:-1], "actual: ", row[-1], "predict: ", self.predict(row[:-1]), "weights: ", self.weights)
            if (self.predict(row[:-1]) != row[-1]):
                error_count += 1
        print("Error count: ", error_count)
        print("Weights: ", self.weights[1:])
        print("Bias weight:", self.weights[0])

&#9989; **Do This:** Test your code with the provided code below. Change the number of iterations, the learning rate, and see what happens. Make sure `errors` prints out the weights as you will need them below. It might be useful to write a loop or two to explore how your number of errors changes as you modify the number of iterations or the learning rate.

In [ ]:
f = open("binary-iris.csv")
header = next(f) # dump the header line
data = []
for line in f:
    fields = line.split(",")
    # need to strip label because, as the last element, it has a \n
    label = (1.0 if fields[2].strip() == "Iris-setosa" else -1.0)
    # the fields are strings until we conver them
    data.append([float(fields[0]), float(fields[1]), label])
f.close()

p = Perceptron(data, 10, 0.01)
p.fit()
p.errors()

---
### 4. Visualizing our results: plotting the decision boundary

At this point, if you feel like your Perceptron class is working as intended, we can try and visualize this.

Once we have run our algorithm, we can try and plot the decision boundary as well as the data entries.

To make this happen, let's do some more math!

Our basic calculation is $w \cdot x + bias = 0 $ where `w` and `x` are vectors and the operation "$\cdot$" is the dot product. Since this is in two dimensions (two features for an input), we can rewrite this as: $w_1  x_1 + w_2  x_2 + b = 0$,
where $b=w_0$.

That looks a lot like an equation of a line in the form of $Ax + By + C = 0$ if we assume $x_1 == x$ and $x_2 == y$.

if we rewrite this, isolating $x_2$, we get:

$$x_2 = \frac{-w_1}{w_2}x_1 - \frac{b}{w_2}$$ 

From that, comparing with the standard form of $y = mx+b$, we can read off our slope and intercept:

$$slope = -\frac{w_1}{w_2}$$

$$intercept = \frac{-b}{w_2}$$

Plug in your weights and plot the line. Plot the data as well where they are colored to indicate which class they belong to.

The data should be stored in the `Perceptron` instance along with the weights (which `errors` should print out) and plot the line. The separate the two classes and plot each as a different color.

In [ ]:
m = -1.0*p.weights[1]/p.weights[2]
b = -1.0*p.weights[0]/p.weights[2]

setosa = [ [x,y] for x,y,label in p.data if label==1]
versa = [ [x,y] for x,y,label in p.data if label==-1]
for x,y in setosa:
    plt.scatter(x,y,c="red")
for x,y in versa:
    plt.scatter(x,y,c="blue")

setosa = np.array(setosa)
versa = np.array(versa)
min_x = min(setosa[:,0].min(),versa[:,0].min())
max_x = max(setosa[:,0].max(),versa[:,0].max())
plt.plot(np.linspace(min_x,max_x,50), [m*x+b for x in np.linspace(min_x,max_x,50)], "k-")

plt.show()